# Unsupervised Structure Discovery (Density Clustering) — Calibration on the ELSST train split

This notebook uses UMAP (optional) and HDBSCAN to group terms. We sweep the number of UMAP dimensions,
`min_cluster_size` and `min_samples` and keep the setting with the highest F1 on the ELSST train split.

In [ ]:
# project paths and the device come from scripts/config.py (works locally and on Colab)
import sys
import os
import gc
from pathlib import Path
import importlib

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts').exists() and (candidate / 'analysis').exists():
            return candidate
    raise RuntimeError("Could not locate project root containing scripts/ and analysis/")

sys.path.insert(0, str(_find_project_root(Path.cwd().resolve())))

import numpy as np
import pandas as pd
import torch
from scripts import model_utils_clustering as model_utils
from scripts import model_utils_shared, config
import warnings

# Reload config to get latest changes
importlib.reload(config)

# Silence warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Reproducibility + deterministic output paths
SEED = model_utils_shared.setup_reproducibility(config.SEED)

PROJECT_ROOT = Path(config.PROJECT_ROOT)
ELSST_CALLIBRATIONS_DIR = Path(config.CALIBRATION_DIR)
ELSST_PLOTS_DIR = Path(config.CALIBRATION_PLOTS_DIR)
ELSST_CALLIBRATIONS_DIR.mkdir(parents=True, exist_ok=True)
ELSST_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Calibration results: {ELSST_CALLIBRATIONS_DIR}")
print(f"Device: {config.DEVICE}")
print(f"Using seed: {SEED}")


## 1. Load Data

In [ ]:
# Calibration data: ELSST train split only (Parquet files with integer term ids).
# Negatives use all labels of both concepts. idx1 / idx2 point into `terms`.
data = model_utils_shared.load_pair_data(**config.DATASETS["elsst_train"])
terms = data["terms"]
idx1, idx2, labels = data["idx1"], data["idx2"], data["label"]
print(f"Total pairs: {len(labels):,} | positives: {int(labels.sum()):,} | unique terms: {len(terms):,}")


## 2. Grid Search per Model

Encode the terms once per model, then sweep the clustering parameters and evaluate every setting.

In [ ]:
# embeddings from model_utils_shared.build_embeddings (model on config.DEVICE);
# evaluation on the integer pair arrays
models_to_test = config.MODELS
all_clustering_results = []

for model_name, model_conf in models_to_test.items():
    display_name = model_conf['display_name']
    print(f"\n{'='*50}")
    print(f" Processing: {display_name}")
    print(f"{'='*50}")

    # 1. Load Model
    model = model_utils_shared.load_model(model_name, model_type=model_conf['type'], device=config.DEVICE)
    if model is None:
        continue

    # 2. Embeddings of all terms (L2-normalised)
    emb = model_utils_shared.build_embeddings(model, terms, batch_size=config.BATCH_SIZE)
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    # 3. Sweep Hyperparameters (UMAP Components + Min Cluster Size + Min Samples)
    results, best_params = model_utils.run_clustering_optimization(
        display_name,
        emb["norm_np"],
        idx1, idx2, labels,
        config.CLUSTERING_PARAMS['min_cluster_sizes'],
        config.CLUSTERING_PARAMS['min_samples'],
        config.CLUSTERING_PARAMS['umap_n_components']
    )

    all_clustering_results.extend(results)
    del emb
    gc.collect()

    # 4. Plot Heatmap
    results_df = pd.DataFrame(results)
    model_utils.plot_clustering_heatmap(results_df, display_name, plots_dir=str(ELSST_PLOTS_DIR))

## 3. Summary of Best Clustering Configurations

In [ ]:
full_results_df = pd.DataFrame(all_clustering_results)

# Print Summary
model_utils.print_best_clustering_settings(full_results_df)

csv_path = ELSST_CALLIBRATIONS_DIR / 'clustering_results.csv'
full_results_df.to_csv(csv_path, index=False)
print(f"\nDetailed results saved to: {csv_path}")

# copy these lines into scripts/config.py
print("\nPaste into scripts/config.py:\n")
print("BEST_CLUSTERING_PARAMS = {")
for model_name, model_conf in config.MODELS.items():
    sub = full_results_df[full_results_df['model'] == model_conf['display_name']]
    if sub.empty:
        continue
    best = sub.loc[sub['f1'].idxmax()]
    print(f'    "{model_name}": {{"n_components": {int(best["n_components"])}, '
          f'"min_cluster_size": {int(best["min_cluster_size"])}, "min_samples": {int(best["min_samples"])}}},')
print("}")